# Hair Painter -- U-Net Training (Kaggle GPU)

Leave-one-out U-Net training for Mimivirus fibril segmentation in TEM images.

**Accelerators supported:** T4 x2 (recommended) | P100 | TPU v3-8

**Required dataset:** `gabrieldias0202/hair-painter` -- add to notebook before running.

**Workflow:**
1. Install dependencies
2. Detect GPU/TPU, configure DataParallel + AMP
3. Read data from attached dataset
4. Train 19 folds (leave-one-out)
5. Save checkpoints to `/kaggle/working/output/unet/`
6. Download the `.pt` files from the Output panel

---
Local baseline (CPU, base=16, 25 epochs): **F1@5px = 0.41** vs 0.31 classical.

In [ ]:
# -- 1. Dependencies --------------------------------------------------------
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# PyTorch 2.5.1+cu118 is the LAST version supporting P100 (sm_60).
# 2.6+ dropped sm_60. Default Kaggle 2.10+cu128 supports sm_70+ only.
pip("torch==2.5.1+cu118",
    "--extra-index-url", "https://download.pytorch.org/whl/cu118")
pip("tifffile>=2024.2.12", "scikit-image>=0.22.0", "scipy>=1.12.0", "imagecodecs")
print("Dependencies OK")

In [ ]:
# -- 2. Accelerator info ----------------------------------------------------
import torch

N_GPU = torch.cuda.device_count()
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}  |  GPUs: {N_GPU}")
for i in range(N_GPU):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB")

In [ ]:
# -- 3. Path setup -- find dataset dynamically (handles UI and CLI mounts) ----
import os, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "output" / "unet"

def find_dataset_root(input_dir=Path("/kaggle/input")):
    """Find dataset root by looking for scripts/train_kaggle.py."""
    def _check(d):
        return d.is_dir() and (d / "scripts" / "train_kaggle.py").exists()
    top = sorted(input_dir.iterdir()) if input_dir.exists() else []
    print(f"  /kaggle/input: {[d.name for d in top]}")
    # Level 0: /kaggle/input/<name>/
    for d in top:
        if _check(d):
            return d
    # Level 1: /kaggle/input/<subdir>/<name>/
    for d in top:
        if d.is_dir():
            for sub in sorted(d.iterdir()):
                if _check(sub):
                    return sub
    # Level 2: /kaggle/input/<subdir>/<user>/<name>/
    for d in top:
        if d.is_dir():
            for sub in sorted(d.iterdir()):
                if sub.is_dir():
                    for subsub in sorted(sub.iterdir()):
                        if _check(subsub):
                            return subsub
    return None

DATASET = find_dataset_root()
if DATASET is None:
    raise FileNotFoundError(
        "Dataset root not found (missing scripts/train_kaggle.py). "
        "Add 'gabrieldias0202/hair-painter' via Data > Add Input."
    )
print(f"Dataset: {DATASET}")

data_link = WORKING / "Data"
if not data_link.exists():
    os.symlink(str(DATASET / "Data"), str(data_link))
    print(f"Linked Data/ -> {DATASET}/Data")
else:
    print(f"Data/ already linked")

if str(DATASET) not in sys.path:
    sys.path.insert(0, str(DATASET))

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUT_DIR}")
raw_count = len(list((DATASET / "Data/Raw").glob("*.tif")))
mask_count = len(list((DATASET / "Data/Manual_paint").glob("*")))
print(f"Raw TIFs: {raw_count} | Masks: {mask_count}")

In [ ]:
# -- 4. Training configuration ----------------------------------------------
# Tune per accelerator:
#
#  Accelerator   | epochs | batch | base | patches/img | estimated time
#  T4 x2 (rec.) |  150   |   64  |  64  |    400      |  ~4-6 h total
#  T4 x1        |  150   |   32  |  64  |    400      |  ~7-9 h total
#  P100         |  150   |   32  |  64  |    400      |  ~6-8 h total
#  TPU v3-8     |  150   |   64  |  64  |    400      |  ~3-4 h total
#  CPU (test)   |    5   |    8  |  16  |     50      |  ~30 min/fold

CFG = dict(
    epochs            = 150,
    patches_per_image = 400,
    batch             = 64,   # DataParallel splits across GPUs automatically
    base              = 64,   # UNet base channels (64 = better quality)
    lr                = 5e-4,
    variant           = "production",
    n_images          = 19,
    checkpoint_every  = 10,
    out               = str(OUT_DIR),
    # Fold control:
    all_folds         = True,  # False = train only the fold below
    val               = 1,     # only used when all_folds=False
)
print("Configuration:")
for k, v in CFG.items():
    print(f"  {k:<22} = {v}")

In [ ]:
# -- 5. Train ---------------------------------------------------------------
import types, time
from scripts.train_kaggle import train_fold_gpu, DEVICE, _USE_TPU

print(f"Device: {DEVICE}")
if _USE_TPU:
    print("Mode: TPU (torch_xla)")
else:
    ngpu = torch.cuda.device_count()
    mode = f"DataParallel {ngpu}x GPU" if ngpu > 1 else f"{ngpu}x GPU"
    print(f"Mode: {mode}")
    print(f"AMP (mixed precision): {DEVICE.type == 'cuda'}")

args = types.SimpleNamespace(**{k: v for k, v in CFG.items() if k not in ("all_folds", "val")})

t0 = time.time()
if CFG["all_folds"]:
    for n in range(1, CFG["n_images"] + 1):
        train_fold_gpu(n, args)
else:
    train_fold_gpu(CFG["val"], args)

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed/3600:.1f} h")

In [ ]:
# -- 6. List checkpoints (for download) -------------------------------------
print(f"Checkpoints in {OUT_DIR}:\n")
ckpts = sorted(OUT_DIR.glob("*.pt"))
total_mb = 0
for f in ckpts:
    mb = f.stat().st_size / 1e6
    total_mb += mb
    print(f"  {f.name:<35} {mb:6.1f} MB")
print(f"\nTotal: {len(ckpts)} files  {total_mb:.0f} MB")
print("\nTo download: click 'Output' in the sidebar -> select .pt files")

In [ ]:
# -- 7. Quick evaluation -- F1 per fold (optional, after training) ----------
# Computes F1@5px and IoU for each fold with the optimal threshold.
# Takes a few minutes (sliding-window inference).

import json
import numpy as np
from PIL import Image

from scripts.train_kaggle   import DEVICE as _DEV
from scripts.train_unet     import UNet, load_gt_mask
from scripts.infer_unet     import predict_full
from scripts.preprocess_variants import VARIANTS
from scripts.metrics        import f1_tolerance, iou, annotated_quadrant_mask
from hairpainter.services.io.io_service import IOService

results = {}
for val_n in range(1, CFG["n_images"] + 1):
    ckpt_path = OUT_DIR / f"unet_fold_val{val_n}.pt"
    if not ckpt_path.exists():
        print(f"fold {val_n}: checkpoint not found, skipping")
        continue

    ck = torch.load(str(ckpt_path), map_location="cpu")
    model = UNet(base=ck.get("base", 32))
    model.load_state_dict(ck["state_dict"])
    variant = ck.get("variant", "production")

    img  = IOService().load(Path(f"Data/Raw/{val_n}.tif"))
    gray = VARIANTS[variant](img.array.copy(), {"image_data": img}).astype(np.float32) / 255.0
    prob = predict_full(model, gray)

    gt = load_gt_mask(val_n).astype(bool)
    if gt.shape != prob.shape:
        pil_gt = Image.fromarray(gt.astype(np.uint8) * 255)
        pil_gt = pil_gt.resize((prob.shape[1], prob.shape[0]), Image.NEAREST)
        gt = np.array(pil_gt).astype(bool)

    quad = annotated_quadrant_mask(gt.shape)
    best = {"f1": -1.0}
    for t in np.linspace(0.1, 0.9, 17):
        pred = prob >= t
        f1   = f1_tolerance(pred, gt, tol=5, quad=quad)
        if f1["f1"] > best["f1"]:
            best = {"t": round(float(t), 3), "f1": round(f1["f1"], 4),
                    "recall": round(f1["recall"], 4),
                    "precision": round(f1["precision"], 4),
                    "iou": round(iou(pred, gt, quad), 4)}

    results[val_n] = best
    print(f"fold {val_n:2d}  F1={best['f1']:.3f}  IoU={best['iou']:.3f}  "
          f"rec={best['recall']:.3f}  prec={best['precision']:.3f}  t={best['t']}")

if results:
    mean_f1  = np.mean([v["f1"]  for v in results.values()])
    mean_iou = np.mean([v["iou"] for v in results.values()])
    print(f"\nMean: F1={mean_f1:.3f}  IoU={mean_iou:.3f}  "
          "(classical baseline F1~0.31, U-Net CPU F1~0.41)")

    report = {"config": CFG, "folds": results,
              "mean": {"f1": round(mean_f1,4), "iou": round(mean_iou,4)}}
    rep_path = OUT_DIR / "cv_report_kaggle.json"
    rep_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"Report saved: {rep_path}")